# 11 — Generate Tables & Figures

**Every manuscript number in this notebook is derived programmatically
from the corrected source files loaded here.** No result is hard-coded
into a cell. Consistency validation compares all 9 freshly-generated tables against
stored reference tables (`supplementary/reference_tables/`) -- themselves
produced by a validated run of this same pipeline -- programmatically,
using a NUMERIC tolerance comparison (not a file-hash checksum; no
checksum manifest is claimed or shipped). Each table has its own stated
tolerance, reflecting its own numerical nature (e.g. p-values need a
tighter tolerance than a bootstrap CI bound). If any generated value
differs from its reference by more than that table's tolerance, this
notebook FAILS loudly; it never silently substitutes a different number.


In [ ]:
import os, sys, json, hashlib
import pandas as pd
import numpy as np
assert 'REPO_ROOT' in dir(), "Run notebook 00 first (or re-execute its setup cells)."
sys.path.insert(0, REPO_ROOT)

RESULTS_DIR = os.path.join(REPO_ROOT, "results")
TABLES_DIR = os.path.join(REPO_ROOT, "tables")
FIGURES_DIR = os.path.join(REPO_ROOT, "figures")
REFERENCE_DIR = os.path.join(REPO_ROOT, "supplementary", "reference_tables")
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

EXP_DIR = os.path.join(REPO_ROOT, "data_deidentified", "experiment_outputs")


## Table 1: SA calibration -- T0/cooling grid

In [ ]:
t0_grid = pd.read_csv(os.path.join(EXP_DIR, "sa_t0_cooling_grid.csv"))
t0_grid.to_csv(os.path.join(TABLES_DIR, "table_sa_t0_cooling_grid.csv"), index=False)
print(t0_grid.to_string(index=False))


## Table 2: SA calibration -- lambda_s sweep

In [ ]:
lambda_sweep = pd.read_csv(os.path.join(EXP_DIR, "sa_lambda_sweep.csv"))
lambda_sweep.to_csv(os.path.join(TABLES_DIR, "table_sa_lambda_sweep.csv"), index=False)
print(lambda_sweep.to_string(index=False))


## Table 3: NR/FR/RG/SA held-out descriptive summary

In [ ]:
main_df = pd.read_csv(os.path.join(EXP_DIR, "nr_fr_rg_sa_block_level.csv"))
nr_dist_total = main_df[main_df['method']=='NR']['total_distance_km'].sum()
rows = []
for m in ['NR','FR','RG','SA']:
    sub = main_df[main_df['method']==m]
    n = len(sub)
    rows.append(dict(method=m, lateness=sub['total_lateness_min'].mean(),
        otd_pct=100*sub['otd_num'].sum()/sub['otd_den'].sum(),
        dist_premium_pct=100*(sub['total_distance_km'].sum()-nr_dist_total)/nr_dist_total if m!='NR' else 0.0,
        rsi=sub['rsi'].mean(), moved_stops=sub['moved_stops'].mean(),
        block_unchanged_pct=100*sub['block_fully_unchanged'].sum()/n if m!='NR' else 100.0,
        no_harm_pct=100*(1-sub['block_any_harm'].sum()/n) if m!='NR' else 100.0))
table3 = pd.DataFrame(rows)
table3.to_csv(os.path.join(TABLES_DIR, "table_nr_fr_rg_sa_summary.csv"), index=False)
print(table3.to_string(index=False))


## Table 4: Fuzzy calibration/robustness summary

In [ ]:
fuzzy_holdout = pd.read_csv(os.path.join(EXP_DIR, "fuzzy_holdout_block_level.csv"))
n = len(fuzzy_holdout)
pos_risk = fuzzy_holdout['total_lateness_min_nr'] > 0.01
always_sa_reduction = (fuzzy_holdout.loc[pos_risk,'total_lateness_min_nr'] - fuzzy_holdout.loc[pos_risk,'total_lateness_min_sa']).sum()
fz_reduction = (fuzzy_holdout.loc[pos_risk,'total_lateness_min_nr'] - fuzzy_holdout['fz_lateness'][pos_risk]).sum()
retention = 100*fz_reduction/always_sa_reduction
table4 = pd.DataFrame([dict(
    activation_rate_pct=100*fuzzy_holdout['activate'].mean(), retention_pct=retention,
    rsi=fuzzy_holdout['fz_rsi'].mean(), moved_stops=fuzzy_holdout['fz_moved'].mean(),
    no_harm_pct=100*(1-(fuzzy_holdout['activate'] & (fuzzy_holdout['fz_lateness']>fuzzy_holdout['total_lateness_min_nr']+1e-6)).sum()/n))])
table4.to_csv(os.path.join(TABLES_DIR, "table_fuzzy_summary.csv"), index=False)
print(table4.to_string(index=False))


## Table 5: Liu-ALNS external-comparator table

In [ ]:
liu_df = pd.read_csv(os.path.join(EXP_DIR, "liu_alns_block_level.csv"))
table5 = pd.DataFrame([dict(
    method='Liu-ALNS', lateness=liu_df['total_lateness_min'].mean(),
    otd_pct=100*liu_df['otd_num'].sum()/liu_df['otd_den'].sum(),
    dist_premium_pct=100*(liu_df['total_distance_km'].sum()-nr_dist_total)/nr_dist_total,
    rsi=liu_df['rsi'].mean(), moved_stops=liu_df['moved_stops'].mean(),
    block_unchanged_pct=100*liu_df['block_fully_unchanged'].sum()/len(liu_df),
    no_harm_pct=100*(1-liu_df['block_any_harm'].sum()/len(liu_df)))])
table5.to_csv(os.path.join(TABLES_DIR, "table_liu_alns_summary.csv"), index=False)
print(table5.to_string(index=False))


## Table 6: no-harm / unchanged, with EXPLICIT denominators (block vs vehicle-level never mixed)

In [ ]:
table6_rows = []
for m in ['FR','RG','SA']:
    sub = main_df[main_df['method']==m]
    table6_rows.append(dict(method=m, level='block', n=len(sub),
        no_harm_pct=100*(1-sub['block_any_harm'].sum()/len(sub)),
        unchanged_pct=100*sub['block_fully_unchanged'].sum()/len(sub)))
table6_rows.append(dict(method='Liu-ALNS', level='block', n=len(liu_df),
    no_harm_pct=100*(1-liu_df['block_any_harm'].sum()/len(liu_df)),
    unchanged_pct=100*liu_df['block_fully_unchanged'].sum()/len(liu_df)))
table6 = pd.DataFrame(table6_rows)
table6.to_csv(os.path.join(TABLES_DIR, "table_no_harm_unchanged.csv"), index=False)
print(table6.to_string(index=False))


## Table 7: daily-instance statistical table (from Notebook 10's saved outputs)

In [ ]:
wilcoxon_path = os.path.join(RESULTS_DIR, "wilcoxon_holm_results.csv")
if os.path.exists(wilcoxon_path):
    table7 = pd.read_csv(wilcoxon_path)
    table7.to_csv(os.path.join(TABLES_DIR, "table_daily_instance_statistics.csv"), index=False)
    print(table7.to_string(index=False))
else:
    print("Run notebook 10_STATISTICAL_ANALYSIS.ipynb first to produce wilcoxon_holm_results.csv.")


## Table 8: ERR TomTom robustness table

In [ ]:
err_summary_path = os.path.join(RESULTS_DIR, "err_tomtom_robustness_summary.json")
if os.path.exists(err_summary_path):
    with open(err_summary_path) as f:
        err_summary = json.load(f)
    table8 = pd.DataFrame([{k: v for k, v in err_summary.items() if k != 'metadata'}])
    table8.to_csv(os.path.join(TABLES_DIR, "table_err_robustness.csv"), index=False)
    print(table8.to_string(index=False))
else:
    print("Run notebook 09_TOMTOM_EMPIRICAL_ROBUSTNESS.ipynb first.")


## Table 9: runtime table

In [ ]:
runtime_rows = []
for m in ['FR','RG','SA']:
    sub = main_df[main_df['method']==m]
    if 'runtime_ms' in sub.columns:
        rt = sub['runtime_ms']
        runtime_rows.append(dict(method=m, mean_ms=rt.mean(), median_ms=rt.median(),
                                  p95_ms=rt.quantile(0.95), max_ms=rt.max()))
if 'runtime_ms' in liu_df.columns:
    rt = liu_df['runtime_ms']
    runtime_rows.append(dict(method='Liu-ALNS', mean_ms=rt.mean(), median_ms=rt.median(),
                              p95_ms=rt.quantile(0.95), max_ms=rt.max()))
table9 = pd.DataFrame(runtime_rows)
table9.to_csv(os.path.join(TABLES_DIR, "table_runtime.csv"), index=False)
print(table9.to_string(index=False))


## Figure: lateness comparison across methods (publication-quality)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

methods = table3['method'].tolist() + ['Liu-ALNS']
lateness_vals = table3['lateness'].tolist() + [table5['lateness'].iloc[0]]

fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
bars = ax.bar(methods, lateness_vals, color=['#888888','#4C72B0','#55A868','#C44E52','#8172B2'])
ax.set_ylabel("Mean lateness (min)")
ax.set_title("Mean lateness by method (51-instance held-out, corrected)")
for bar, val in zip(bars, lateness_vals):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.5, f"{val:.1f}", ha='center', fontsize=9)
plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "fig_lateness_comparison.png")
plt.savefig(fig_path)
plt.savefig(fig_path.replace('.png', '.pdf'))
plt.close()
print(f"Saved {fig_path} (+ .pdf)")


## Reference-table consistency validation (programmatic, no literal numbers)

Compares all 9 freshly-generated tables above against a stored reference
table in `supplementary/reference_tables/` -- itself produced by a
validated run of this identical pipeline, compared via numeric tolerance
(not a file checksum). This checks REPRODUCIBILITY (does re-running the pipeline on the same source
data reproduce the same numbers?), not agreement with any number written
directly into this notebook's code.

If `supplementary/reference_tables/` does not contain a reference for a
given generated table (e.g. this is the very first validated run, used
to *establish* the references), that table is skipped with a clear note
rather than silently treated as passing.


In [ ]:
GENERATED_TABLES = {
    "table_sa_t0_cooling_grid.csv": t0_grid,
    "table_sa_lambda_sweep.csv": lambda_sweep,
    "table_nr_fr_rg_sa_summary.csv": table3,
    "table_fuzzy_summary.csv": table4,
    "table_liu_alns_summary.csv": table5,
    "table_no_harm_unchanged.csv": table6,
    "table_runtime.csv": table9,
}
if 'table7' in dir():
    GENERATED_TABLES["table_daily_instance_statistics.csv"] = table7
if 'table8' in dir():
    GENERATED_TABLES["table_err_robustness.csv"] = table8

# Table-specific numerical tolerances. All tables EXCEPT table_runtime.csv
# are computed deterministically from STATIC, already-stored source CSVs
# (loaded and aggregated with fixed seeds where any randomness is
# involved, e.g. the bootstrap in notebook 10) -- re-running this pipeline
# on the same source data should reproduce IDENTICAL values, so a tight
# floating-point tolerance is used, not a loose "close enough" band.
# table_runtime.csv is the sole exception: it reports actual measured
# wall-clock timings from a real computation, which legitimately vary
# with machine/hardware load across runs.
DETERMINISTIC_TOLERANCE = 1e-6
TABLE_TOLERANCES = {
    "table_sa_t0_cooling_grid.csv": DETERMINISTIC_TOLERANCE,
    "table_sa_lambda_sweep.csv": DETERMINISTIC_TOLERANCE,
    "table_nr_fr_rg_sa_summary.csv": DETERMINISTIC_TOLERANCE,
    "table_fuzzy_summary.csv": DETERMINISTIC_TOLERANCE,
    "table_liu_alns_summary.csv": DETERMINISTIC_TOLERANCE,
    "table_no_harm_unchanged.csv": DETERMINISTIC_TOLERANCE,
    "table_runtime.csv": 1.0,          # runtime (ms) has genuine cross-run/hardware variation
    "table_daily_instance_statistics.csv": DETERMINISTIC_TOLERANCE,
    "table_err_robustness.csv": DETERMINISTIC_TOLERANCE,
}

def compare_tables(generated_df, reference_df, tol):
    if generated_df.shape != reference_df.shape:
        return False, f"shape mismatch: generated {generated_df.shape} vs reference {reference_df.shape}"
    numeric_cols = generated_df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if col not in reference_df.columns:
            return False, f"column '{col}' missing from reference"
        diff = (generated_df[col].values - reference_df[col].values)
        max_abs_diff = np.nanmax(np.abs(diff)) if len(diff) else 0.0
        if max_abs_diff > tol:
            return False, f"column '{col}' max abs diff {max_abs_diff:.4g} exceeds tolerance {tol}"
    non_numeric_cols = [c for c in generated_df.columns if c not in numeric_cols]
    for col in non_numeric_cols:
        if col in reference_df.columns and not generated_df[col].astype(str).equals(reference_df[col].astype(str)):
            return False, f"non-numeric column '{col}' differs"
    return True, "within tolerance"

consistency_rows = []
os.makedirs(REFERENCE_DIR, exist_ok=True)
for fname, gen_df in GENERATED_TABLES.items():
    tol = TABLE_TOLERANCES[fname]
    ref_path = os.path.join(REFERENCE_DIR, fname)
    if not os.path.exists(ref_path):
        consistency_rows.append(dict(table=fname, status="NO_REFERENCE", tolerance=tol,
                                      detail="No stored reference found -- cannot validate reproducibility for this table."))
        continue
    ref_df = pd.read_csv(ref_path)
    ok, detail = compare_tables(gen_df, ref_df, tol)
    consistency_rows.append(dict(table=fname, status="PASS" if ok else "FAIL", tolerance=tol, detail=detail))

consistency_df = pd.DataFrame(consistency_rows)
print(consistency_df.to_string(index=False))
consistency_df.to_csv(os.path.join(RESULTS_DIR, "table_generation_consistency_audit.csv"), index=False)

n_tables_expected = 9
any_fail = (consistency_df['status'] == "FAIL").any()
n_no_reference = (consistency_df['status'] == "NO_REFERENCE").sum()
print(f"\nTables validated: {len(consistency_df)}/{n_tables_expected} expected")
print(f"{'FAIL' if any_fail else 'PASS'} -- reference-table consistency "
      f"({n_no_reference} table(s) had no stored reference to compare against)")
if len(consistency_df) < n_tables_expected:
    print(f"WARNING: only {len(consistency_df)}/{n_tables_expected} tables were available to check "
          f"(table7/table8 require notebooks 10 and 09 to have run first in this session).")
if any_fail:
    raise AssertionError(
        "One or more generated tables do NOT match their stored reference "
        "within tolerance. Investigate the ROOT CAUSE -- do not edit the "
        "generated numbers manually, and do not overwrite the reference "
        "table to force a pass.")


## (One-time) establish reference tables

Run this cell ONLY when deliberately re-baselining the reference tables
after a genuine, reviewed change to the corrected source data or
pipeline (never to silently paper over a consistency failure above).
Disabled by default.


In [ ]:
ESTABLISH_NEW_REFERENCES = False  # deliberately off by default

if ESTABLISH_NEW_REFERENCES:
    os.makedirs(REFERENCE_DIR, exist_ok=True)
    for fname, gen_df in GENERATED_TABLES.items():
        gen_df.to_csv(os.path.join(REFERENCE_DIR, fname), index=False)
    print(f"Wrote {len(GENERATED_TABLES)} reference tables to {REFERENCE_DIR}")
else:
    print("ESTABLISH_NEW_REFERENCES is False -- reference tables left unchanged.")


## Expected outputs / integrity checks

In [ ]:
checks = {
    "all_generated_tables_written": all(os.path.exists(os.path.join(TABLES_DIR, f)) for f in GENERATED_TABLES.keys()),
    "all_9_tables_present": len(GENERATED_TABLES) == 9,
    "figure_written": os.path.exists(os.path.join(FIGURES_DIR, "fig_lateness_comparison.png")),
    "reference_consistency_no_failures": not any_fail,
}
for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
NOTEBOOK_11_STATUS = "PASS" if all(checks.values()) else "FAIL"
print(f"\nNOTEBOOK 11 STATUS: {NOTEBOOK_11_STATUS}")
assert NOTEBOOK_11_STATUS == "PASS"
